In [4]:
import os
import glob
import pandas as pd

import rasterio
import numpy as np

In [5]:
DATA_PATH = "/content/drive/MyDrive/collab/f2/india+portugal+historical.csv"
HISTORICAL_FIRE_PATH = "/content/drive/MyDrive/collab/f2/grid_with_population.csv"

In [6]:
# ── Chargement ────────────────────────────────────────────────────────────────
# Décommenter pour Colab :
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

data = pd.read_csv(DATA_PATH)
print(f"Shape         : {data.shape}")
print(f"\nValeurs manquantes par feature :")

data.head(3)

Mounted at /content/drive
Shape         : (89198, 50)

Valeurs manquantes par feature :


,latitude,longitude,landcover,COARSE,SAND,SILT,CLAY,TEXTURE_USDA,TEXTURE_SOTER,BULK,...,mean_aspect,mean_slope,mean_twi,ndvi_final,was_in_fire_2021,was_in_fire_2021_distance,was_in_fire_2022,was_in_fire_2022_distance,was_in_fire_2023,was_in_fire_2023_distance
0,36.987823,-7.963676,210.0,3.0,26.0,33.0,41.0,3.0,F,1.06,...,196.846283,0.571321,6.447956,0.686275,0.0,4.663419,0.0,5.071836,0.0,10.377242
1,36.994129,-8.952213,110.0,5.0,20.0,35.0,45.0,3.0,F,1.55,...,178.080078,2.499209,3.489216,0.866667,0.0,19.689417,0.0,11.433553,0.0,22.722530
2,36.996850,-7.974890,210.0,3.0,26.0,33.0,41.0,3.0,F,1.06,...,179.751862,1.402498,7.057358,0.889725,0.0,3.958900,0.0,3.870817,0.0,9.577471


In [7]:
distance = pd.read_csv(HISTORICAL_FIRE_PATH)
print(f"Shape         : {distance.shape}")

Shape         : (89198, 4)


In [8]:
data.columns

Index(['latitude', 'longitude', 'landcover', 'COARSE', 'SAND', 'SILT', 'CLAY',
       'TEXTURE_USDA', 'TEXTURE_SOTER', 'BULK', 'ORG_CARBON', 'PH_WATER',
       'CN_RATIO', 'CEC_SOIL', 'CEC_CLAY', 'CEC_EFF', 'BSAT', 'ALUM_SAT',
       'ESP', 'TCARBON_EQ', 'GYPSUM', 'ELEC_COND', 'in_fire', 'prec_ete',
       'tmax_ete', 'tmax_hiver', 'tmin_ete', 'Elevation',
       'dif_perc_hiver_printemps', 'dif_perc_automne_printemps',
       'dif_perc_ete_automne', 'dif_max_automne_printemps',
       'dif_max_hiver_printemps', 'dif_min_hiver_printemps',
       'dif_min_automne_hiver', 'dif_max_hiver_min_automne', 'dist_eau_m',
       'dist_residentiel_m', 'dist_route_m', 'mean_LST_celsius', 'mean_aspect',
       'mean_slope', 'mean_twi', 'ndvi_final', 'was_in_fire_2021',
       'was_in_fire_2021_distance', 'was_in_fire_2022',
       'was_in_fire_2022_distance', 'was_in_fire_2023',
       'was_in_fire_2023_distance'],
      dtype='object')

In [9]:
distance.columns

Index(['latitude', 'longitude', 'landcover', 'population'], dtype='object')

In [10]:
from sklearn.neighbors import BallTree

seuil_km = 1.0
seuil_rad = seuil_km / 6371.0  


all_rad = np.radians(data[["latitude", "longitude"]].values)
climat_rad = np.radians(distance[["latitude", "longitude"]].values)

tree = BallTree(climat_rad, metric="haversine")


dist, idx = tree.query(all_rad, k=1)


dist_km = dist[:, 0] * 6371.0


distance_cols = distance.columns.difference(["longitude", "latitude"])
merged = data.copy()
merged[distance_cols] = distance.iloc[idx[:, 0]][distance_cols].values

merged["dist_distance_km"] = dist_km
merged["match_distance "] = (dist[:, 0] <= seuil_rad).astype(int)


print("Total all_points:", len(data))
print("Match <= seuil:", merged["match_distance "].sum())
print("No match:", (merged["match_distance "] == 0).sum())
print("Distance min/max (km):", merged["dist_distance_km"].min(), merged["dist_distance_km"].max())


merged.loc[merged["match_distance "] == 0, distance_cols] = np.nan

merged.drop(columns=["dist_distance_km", "match_distance "], inplace=True)

Total all_points: 89198
Match <= seuil: 89198
No match: 0
Distance min/max (km): 0.0 0.0


In [11]:
distance.columns

Index(['latitude', 'longitude', 'landcover', 'population'], dtype='object')

In [12]:
merged.columns

Index(['latitude', 'longitude', 'landcover', 'COARSE', 'SAND', 'SILT', 'CLAY',
       'TEXTURE_USDA', 'TEXTURE_SOTER', 'BULK', 'ORG_CARBON', 'PH_WATER',
       'CN_RATIO', 'CEC_SOIL', 'CEC_CLAY', 'CEC_EFF', 'BSAT', 'ALUM_SAT',
       'ESP', 'TCARBON_EQ', 'GYPSUM', 'ELEC_COND', 'in_fire', 'prec_ete',
       'tmax_ete', 'tmax_hiver', 'tmin_ete', 'Elevation',
       'dif_perc_hiver_printemps', 'dif_perc_automne_printemps',
       'dif_perc_ete_automne', 'dif_max_automne_printemps',
       'dif_max_hiver_printemps', 'dif_min_hiver_printemps',
       'dif_min_automne_hiver', 'dif_max_hiver_min_automne', 'dist_eau_m',
       'dist_residentiel_m', 'dist_route_m', 'mean_LST_celsius', 'mean_aspect',
       'mean_slope', 'mean_twi', 'ndvi_final', 'was_in_fire_2021',
       'was_in_fire_2021_distance', 'was_in_fire_2022',
       'was_in_fire_2022_distance', 'was_in_fire_2023',
       'was_in_fire_2023_distance', 'population'],
      dtype='object')

In [13]:
merged.to_csv("/content/drive/MyDrive/collab/f2/india+portugal+historical+population.csv", index=False)